In [0]:
print("Spark version:", spark.version)
spark.range(5).show()



Spark version: 4.1.0
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA manufacturing")

# drop all old tables so the new schema can be created cleanly
for t in ["bronze_dim_machines","bronze_dim_products","bronze_mes_runs","bronze_sensors",
          "bronze_quality","silver_mes_runs","silver_run_oee","gold_oee_by_machine",
          "gold_daily_oee","gold_machine_rank","gold_weekly_oee","gold_machine_health"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("Old tables dropped. Now re-run Cell 1 through Cell 6.")

Old tables dropped. Now re-run Cell 1 through Cell 6.


In [0]:
from pyspark.sql import functions as F, Window
import random, uuid

spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS manufacturing")
spark.sql("USE SCHEMA manufacturing")

# --- scaled master data: 50 machines, 8 products ---
LINES=[f"LINE-{c}" for c in "ABCDE"]; TYPES=["CNC","Press","Welder","Assembler","Lathe"]
MACHINES=[{"machine_id":f"M{i:02d}","machine_name":f"MC-{i:02d}","line":random.choice(LINES),
           "machine_type":random.choice(TYPES),"maintenance_tier":random.choice(["standard","premium"])}
          for i in range(1,51)]
PRODUCTS=[{"product_id":f"P{n}","product_name":f"Prod-{n}","ideal_cycle_time_sec":c}
          for n,c in [("100",12.0),("200",30.0),("300",45.0),("400",20.0),("500",18.0),("600",25.0),("700",9.0),("800",50.0)]]
MIDS=[m["machine_id"] for m in MACHINES]; PIDS=[p["product_id"] for p in PRODUCTS]
IDEAL={p["product_id"]:p["ideal_cycle_time_sec"] for p in PRODUCTS}

# --- MES runs: 50 machines x 90 days x 3 shifts = 13,500 runs ---
from datetime import datetime, timedelta, timezone
runs=[]; start=datetime.now(timezone.utc).date()-timedelta(days=89)
for d in range(90):
    day=(start+timedelta(days=d)).isoformat()
    for mid in MIDS:
        for shift in ["morning","afternoon","night"]:
            pid=random.choice(PIDS); planned=480; downtime=random.randint(5,120); rt=planned-downtime
            total=int(((rt*60)/IDEAL[pid])*random.uniform(0.70,0.98)); good=total-int(total*random.uniform(0.01,0.09))
            runs.append((str(uuid.uuid4()),day,mid,pid,shift,planned,downtime,total,good))
runs_df=(spark.createDataFrame(runs,["run_id","run_date","machine_id","product_id","shift",
         "planned_production_min","downtime_min","total_units","good_units"]).withColumn("run_date",F.to_date("run_date")))

spark.createDataFrame(MACHINES).write.format("delta").mode("overwrite").saveAsTable("bronze_dim_machines")
spark.createDataFrame(PRODUCTS).write.format("delta").mode("overwrite").saveAsTable("bronze_dim_products")
runs_df.write.format("delta").mode("overwrite").saveAsTable("bronze_mes_runs")
print("Bronze MES + dims written. Runs:", runs_df.count())

Bronze MES + dims written. Runs: 13500


In [0]:
# Read the REAL AI4I CSV from your Unity Catalog Volume, clean column names
cols=["udi","product_id_ai","type","air_temp_k","process_temp_k","rot_speed","torque",
      "tool_wear","machine_failure","twf","hdf","pwf","osf","rnf"]
ai4i=spark.read.option("header",True).option("inferSchema",True).csv(
      "/Volumes/workspace/manufacturing/seed/ai4i2020.csv").toDF(*cols)

rand_machine=F.concat(F.lit("M"),F.lpad((F.floor(F.rand()*50)+1).cast("int").cast("string"),2,"0"))

# Map real AI4I columns -> our sensor schema, replicate x200 to reach ~2M rows
base=ai4i.select(
    F.round(F.col("process_temp_k")-273.15,1).alias("temperature_c"),  # real, K->C
    F.col("rot_speed").cast("int").alias("rotational_speed_rpm"),        # real
    F.round(F.col("torque"),1).alias("torque_nm"),                       # real
    F.col("tool_wear").cast("int").alias("tool_wear_min"),               # real
    F.col("machine_failure").cast("int").alias("machine_failure"))       # real
sensors=(base.crossJoin(spark.range(200))                                # 10k x 200 = ~2M
    .withColumn("machine_id",rand_machine)
    .withColumn("reading_date",F.date_sub(F.current_date(),(F.rand()*90).cast("int")))
    .drop("id"))
sensors.write.format("delta").mode("overwrite").saveAsTable("bronze_sensors")
print("Bronze sensors (AI4I-seeded):", sensors.count())

# Quality inspections at scale (200k)
quality=(spark.range(200000)
    .withColumn("machine_id",rand_machine)
    .withColumn("product_id",F.element_at(F.array(*[F.lit(p) for p in PIDS]),(F.floor(F.rand()*8)+1).cast("int")))
    .withColumn("result",F.when(F.rand()<0.06,"FAIL").otherwise("PASS"))
    .withColumn("inspected_date",F.date_sub(F.current_date(),(F.rand()*90).cast("int"))).drop("id"))
quality.write.format("delta").mode("overwrite").saveAsTable("bronze_quality")
print("Bronze quality:", quality.count())

Bronze sensors (AI4I-seeded): 2000000
Bronze quality: 200000


In [0]:
valid=(F.col("product_id").isNotNull()&(F.col("total_units")>0)&(F.col("good_units")<=F.col("total_units")))
spark.table("bronze_mes_runs").filter(valid).write.format("delta").mode("overwrite").saveAsTable("silver_mes_runs")

r=(spark.table("silver_mes_runs")
   .join(spark.table("bronze_dim_products").select("product_id","ideal_cycle_time_sec"),"product_id","left")
   .withColumn("run_time_min",F.col("planned_production_min")-F.col("downtime_min"))
   .withColumn("availability",F.col("run_time_min")/F.col("planned_production_min"))
   .withColumn("performance",F.least((F.col("ideal_cycle_time_sec")*F.col("total_units"))/(F.col("run_time_min")*60.0),F.lit(1.0)))
   .withColumn("quality",F.col("good_units")/F.col("total_units"))
   .withColumn("oee",F.col("availability")*F.col("performance")*F.col("quality"))
   .withColumn("scrap_units",F.col("total_units")-F.col("good_units")))
r.write.format("delta").mode("overwrite").saveAsTable("silver_run_oee")
print("Silver + per-run OEE written.")

Silver + per-run OEE written.


In [0]:
r=spark.table("silver_run_oee")

# daily OEE per machine
daily=(r.groupBy("machine_id","run_date").agg(F.round(F.avg("oee"),3).alias("daily_oee"),
        F.sum("downtime_min").alias("downtime_min"),F.sum("scrap_units").alias("scrap"),F.sum("total_units").alias("units")))

# WINDOW FUNCTIONS: rolling 7-day avg OEE + running cumulative downtime
w7=Window.partitionBy("machine_id").orderBy("run_date").rowsBetween(-6,0)
wcum=Window.partitionBy("machine_id").orderBy("run_date").rowsBetween(Window.unboundedPreceding,0)
daily_win=(daily.withColumn("rolling_7d_oee",F.round(F.avg("daily_oee").over(w7),3))
                .withColumn("running_downtime",F.sum("downtime_min").over(wcum)))
daily_win.write.format("delta").mode("overwrite").saveAsTable("gold_daily_oee")

# WINDOW: rank machines by OEE within each line
mo=(r.groupBy("machine_id").agg(F.round(F.avg("oee"),3).alias("avg_oee"),F.sum("downtime_min").alias("total_downtime"))
     .join(spark.table("bronze_dim_machines").select("machine_id","line"),"machine_id","left"))
ranked=mo.withColumn("rank_in_line",F.rank().over(Window.partitionBy("line").orderBy(F.desc("avg_oee"))))
ranked.write.format("delta").mode("overwrite").saveAsTable("gold_machine_rank")

# weekly trend (time-series)
weekly=(daily.withColumn("year",F.year("run_date")).withColumn("week",F.weekofyear("run_date"))
        .groupBy("year","week").agg(F.round(F.avg("daily_oee"),3).alias("weekly_oee")).orderBy("year","week"))
weekly.write.format("delta").mode("overwrite").saveAsTable("gold_weekly_oee")

print("Gold window + time-series marts written.")
display(daily_win.filter("machine_id='M01'").orderBy("run_date").select("run_date","daily_oee","rolling_7d_oee","running_downtime"))

Gold window + time-series marts written.


run_date,daily_oee,rolling_7d_oee,running_downtime
2026-05-07,0.734,0.734,142
2026-05-08,0.734,0.734,358
2026-05-09,0.634,0.701,619
2026-05-10,0.679,0.695,848
2026-05-11,0.609,0.678,1170
2026-05-12,0.705,0.683,1412
2026-05-13,0.709,0.686,1550
2026-05-14,0.743,0.688,1648
2026-05-15,0.711,0.684,1790
2026-05-16,0.721,0.697,1988


In [0]:
from pyspark.sql import functions as F

sensor_m=(spark.table("bronze_sensors").groupBy("machine_id").agg(
    F.round(F.avg("temperature_c"),1).alias("avg_temp_c"),
    F.round(F.avg("tool_wear_min"),1).alias("avg_tool_wear"),
    F.sum("machine_failure").alias("sensor_failures")))
quality_m=(spark.table("bronze_quality").groupBy("machine_id").agg(
    F.round(F.avg(F.when(F.col("result")=="FAIL",1.0).otherwise(0.0)),3).alias("fail_rate")))
mes_m=(spark.table("silver_run_oee").groupBy("machine_id").agg(
    F.round(F.avg("oee"),3).alias("avg_oee"),
    F.round(F.sum("scrap_units")/F.sum("total_units"),3).alias("scrap_rate")))

health=(spark.table("bronze_dim_machines").select("machine_id","machine_name","line")
        .join(mes_m,"machine_id","left").join(sensor_m,"machine_id","left").join(quality_m,"machine_id","left"))
health.write.format("delta").mode("overwrite").saveAsTable("gold_machine_health")

corr=health.stat.corr("avg_tool_wear","scrap_rate")
print(f"Correlation (avg tool wear vs scrap rate): {corr:.3f}")
display(health)

Correlation (avg tool wear vs scrap rate): -0.041


machine_id,machine_name,line,avg_oee,scrap_rate,avg_temp_c,avg_tool_wear,sensor_failures,fail_rate
M01,MC-01,LINE-A,0.693,0.05,36.9,107.7,1416,0.066
M02,MC-02,LINE-C,0.687,0.053,36.9,108.2,1325,0.053
M03,MC-03,LINE-C,0.696,0.046,36.9,108.0,1334,0.065
M04,MC-04,LINE-B,0.688,0.05,36.9,107.7,1328,0.059
M05,MC-05,LINE-E,0.688,0.05,36.9,108.2,1329,0.057
M06,MC-06,LINE-A,0.695,0.047,36.9,107.7,1355,0.058
M07,MC-07,LINE-E,0.699,0.05,36.9,108.0,1335,0.065
M08,MC-08,LINE-B,0.702,0.048,36.9,108.3,1304,0.062
M09,MC-09,LINE-E,0.685,0.05,36.9,108.2,1433,0.068
M10,MC-10,LINE-D,0.695,0.051,36.9,107.7,1364,0.062


In [0]:
import uuid
from pyspark.sql import functions as F

new=[(row["run_id"], row["run_date"], row["machine_id"], row["product_id"], row["shift"],
      480, 30, row["total_units"]+50, row["good_units"]+40)
     for row in spark.table("silver_mes_runs").limit(3).collect()]
new+=[(str(uuid.uuid4()), None, "M01","P100","morning",480,20,900,880) for _ in range(2)]
new_df=spark.createDataFrame(new, spark.table("silver_mes_runs").columns)
new_df.createOrReplaceTempView("updates")

before=spark.table("silver_mes_runs").count()
spark.sql("""
MERGE INTO silver_mes_runs t USING updates u ON t.run_id = u.run_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
after=spark.table("silver_mes_runs").count()
print(f"MERGE done. Rows before: {before}, after: {after} (updated 3, inserted 2).")

MERGE done. Rows before: 13500, after: 13502 (updated 3, inserted 2).


In [0]:
from pyspark.sql import functions as F

# First run: initialize the history dimension (all machines current)
if not spark.catalog.tableExists("dim_machine_scd2"):
    (spark.table("bronze_dim_machines")
        .withColumn("valid_from", F.current_date())
        .withColumn("valid_to", F.lit(None).cast("date"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta").saveAsTable("dim_machine_scd2"))
    print("SCD2 initialized with all machines current.")

# Simulate a change: upgrade M01 to premium
updates=(spark.table("bronze_dim_machines")
         .withColumn("maintenance_tier",
                     F.when(F.col("machine_id")=="M01", F.lit("premium"))
                      .otherwise(F.col("maintenance_tier"))))
updates.createOrReplaceTempView("machine_updates")

# Step 1: expire the old current row where a tracked attribute changed
spark.sql("""
MERGE INTO dim_machine_scd2 t
USING machine_updates u
ON t.machine_id = u.machine_id AND t.is_current = true
WHEN MATCHED AND (t.maintenance_tier <> u.maintenance_tier OR t.line <> u.line)
  THEN UPDATE SET t.is_current = false, t.valid_to = current_date()
""")

# Step 2: insert the new current version for the changed machine(s)
spark.sql("""
INSERT INTO dim_machine_scd2
SELECT u.*, current_date() AS valid_from, CAST(NULL AS date) AS valid_to, true AS is_current
FROM machine_updates u
JOIN dim_machine_scd2 t
  ON u.machine_id = t.machine_id AND t.is_current = false AND t.valid_to = current_date()
""")

print("SCD2 MERGE applied — M01 should now have two versions.")
display(spark.sql("SELECT * FROM dim_machine_scd2 WHERE machine_id='M01' ORDER BY valid_from"))

SCD2 initialized with all machines current.
SCD2 MERGE applied — M01 should now have two versions.


line,machine_id,machine_name,machine_type,maintenance_tier,valid_from,valid_to,is_current
LINE-A,M01,MC-01,CNC,standard,2026-08-04,2026-08-04,false
LINE-A,M01,MC-01,CNC,premium,2026-08-04,null,true
